# Firms cleaning

Builds three lists of firms for the `firms` table from the client's vendor export,
their cage maintenance table, and our internal cages table:

1. **Linked vendor-manufacturers**: a vendor row followed by the manufacturers it represents, grouped by `link_id`
2. **Orphan vendors**: vendors with no manufacturer linked to them (they represent themselves)
3. **Orphan manufacturers**: manufacturers with no vendor linked to them

Every list has `company_name, cage, email, phone, contact_name` (all nullable).
The linked list also has `link_id` and `role` (`vendor` or `manufacturer`).

Rows that get dropped along the way are written to `output/rejected.csv` with a reason,
so nothing disappears silently.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

DATA = "data"
OUT = "output"
VENDOR_FILE = f"{DATA}/Sales Patriot Project Vendor File_ANONYMIZED.csv"
MAINT_FILE = f"{DATA}/Cage Maintenance_ANONYMIZED.csv"
CAGES_FILE = f"{DATA}/cages_ANONYMIZED.csv"

FIRM_COLUMNS = ["company_name", "cage", "email", "phone", "contact_name"]
LINKED_COLUMNS = FIRM_COLUMNS + ["link_id", "role"]

## 1. Load

Everything is read as text so codes like `000123` keep their leading zeros.
The cages table is about 590 MB and 4 million rows, so only the three columns we need are loaded.

In [2]:
vendors_raw = pd.read_csv(VENDOR_FILE, dtype=str, low_memory=False)
maint_raw = pd.read_csv(MAINT_FILE, dtype=str)
cages_raw = pd.read_csv(CAGES_FILE, dtype=str, usecols=["id", "company", "status"])

for name, df in [("vendors", vendors_raw), ("cage maintenance", maint_raw), ("cages", cages_raw)]:
    print(f"{name:17} {df.shape[0]:>9,} rows  {df.shape[1]:>4} columns")

vendors              18,318 rows   139 columns
cage maintenance     20,385 rows     5 columns
cages             4,005,840 rows     3 columns


## 2. Inspect the raw files

Before cleaning anything, look at what each file actually contains and how they connect.

### Vendor file
139 columns, most of them almost empty. The ones that matter for a firm are the id, name,
phone, contact and email. There is **no cage code column**, so a vendor's cage has to be found
by matching its name against the cages table.

In [3]:
filled = vendors_raw.notna().sum().sort_values(ascending=False)
print("columns more than half filled:", len(filled[filled > len(vendors_raw) / 2]), "of", len(filled))
print("any column mentioning cage:", [c for c in vendors_raw.columns if "cage" in c.lower()])

vendor_cols = ["VendorId", "VendorName", "Phone", "Contact", "EmailAddress", "Sts"]
vendors_raw[vendor_cols].head(10)

columns more than half filled: 29 of 139
any column mentioning cage: []


,VendorId,VendorName,Phone,Contact,EmailAddress,Sts
0,008223,DO NOT USE,455-5597,ARTHUR SIMPLINA,NaN,T
1,008224,YORK INDUSTRIES,800-964-7035,NaN,NaN,T
2,008225,BILL RAPP SUBARU,626-239-8066,Jaimie & Wade Scarpitti,NaN,T
3,008226,EPICOR SOFTWARE CORP.,602-595-6430,Michael Duncan,sales@partexmarker.com,T
4,008227,ONONDAGA COUNTY CLERK,866-802-6383,NaN,NaN,T
5,008228,J & J INDUSTRIES INC.,704-509-4328,NaN,NaN,T
6,008229,PREFERRED POWER IND.,559-688-2911,sales,NaN,T
7,008230,DAYTON WALTHER,800-521-4888,SCOTT HINDLEY,NaN,T
8,008231,AUSTIN HARDWARE,614-228-5781,Brian Wider,NaN,T
9,008232,"JAGUAR INDUSTRIES, INC.",800-433-0038,JAMIE BLAKE,NaN,T


In [4]:
vendors_raw[vendor_cols].notna().sum().to_frame("non-null")

,non-null
VendorId,18318
VendorName,18292
Phone,17032
Contact,14063
EmailAddress,8363
Sts,18302


In [5]:
# ids are zero-padded text, a few are not numbers at all
print(vendors_raw["VendorId"].sample(5, random_state=1).tolist())
print(vendors_raw.loc[pd.to_numeric(vendors_raw["VendorId"], errors="coerce").isna(), ["VendorId", "VendorName"]])

['111519', '102909', '108696', '111747', '106973']
      VendorId           VendorName
622     00TEST         PRELUDE TEST
1912    00000M                  NaN
2322    000PSI  PRELUDE TEST VENDOR
18317  REPORTS                  NaN


In [6]:
# names that say the vendor should not be used, and exact duplicate names
junk = vendors_raw["VendorName"].str.contains("DO NOT USE|INACTIVE", case=False, na=False)
print("do not use / inactive:", junk.sum())
print("duplicate names:", vendors_raw["VendorName"].str.upper().str.strip().duplicated().sum())
vendors_raw.loc[junk, "VendorName"].head(8).tolist()

do not use / inactive: 50
duplicate names: 285


['DO NOT USE',
 'DO NOT USE. SEE V#101020',
 'DO NOT USE DUP VENDOR SEE ',
 'INACTIVE UNITED PARCEL SERVICE',
 'INACTIVE UNITED PARCEL SERVICE',
 'FLEETPRIDE  *DO NOT USE*',
 'DO NOT USE SEE 3778',
 'INACTIVE UPS SUPPLY CHAIN']

### Cage maintenance
One row per manufacturer cage code. `VendorCode` links the manufacturer to a vendor, and
`Description` is a free-text note: mostly emails, sometimes fax numbers or remarks.

In [7]:
maint_raw.head(10)

,CageCode,Description,VendorCode,PreludeKey,ExportDate
0,NaN,SALES@MILEHIMACHINE.COM,2538.0,001!,2024-07-09 16:15:44.000
1,/tai,NaN,NaN,001! /tai,2024-07-09 16:15:44.000
2,03038,NaN,NaN,001! 03038,2024-07-09 16:15:44.000
3,07690,NaN,NaN,001! 07690,2024-07-09 16:15:44.000
4,0ACP0,DIST,NaN,001! 0ACP0,2024-07-09 16:15:44.000
5,0KXH9,bhurley@spi-co.com,NaN,001! 0KXH9,2024-07-09 16:15:44.000
6,18863,NaN,NaN,001! 18863,2024-07-09 16:15:44.000
7,19522,ESAB DIST,NaN,001! 19522,2024-07-09 16:15:44.000
8,1CRL3,dcunningham@marshbellofram.com,NaN,001! 1CRL3,2024-07-09 16:15:44.000
9,1M7N0,cmccoy@krayden.com,NaN,001! 1M7N0,2024-07-09 16:15:44.000


In [8]:
maint_raw.notna().sum().to_frame("non-null")

,non-null
CageCode,20384
Description,16784
VendorCode,11982
PreludeKey,20385
ExportDate,20385


In [9]:
# cage codes should be 5 characters; some are not
maint_raw["CageCode"].str.strip().str.len().value_counts().sort_index()

CageCode
1.0        4
2.0        3
3.0        8
4.0      137
5.0    20108
6.0      124
Name: count, dtype: int64

In [10]:
desc = maint_raw["Description"].dropna()
print("descriptions with an email:", desc.str.contains("@").sum(), "of", len(desc))
desc[~desc.str.contains("@")].sample(10, random_state=4).tolist()

descriptions with an email: 12906 of 16784


['CURTIS (F) 440-446-0555',
 'ARMOR (F)904-807-5328',
 'LIPHATECH (F) 414-247-8166',
 'LOUISVLLE LADDER ',
 'DO NOT USE ',
 'KWIK FILE (F) 763-572-0168',
 'PA LEONE&SONS(F)3154721824',
 'WILL NOT QUOTE W/OUT SERIAL#',
 'NOSHOK',
 'V# 6997  ']

`VendorCode` is stored as a float (`2538.0`) while `VendorId` is zero-padded text (`002538`),
so the two only match once both are converted to numbers.

In [11]:
vendor_ids = pd.to_numeric(vendors_raw["VendorId"], errors="coerce")
vendor_codes = pd.to_numeric(maint_raw["VendorCode"], errors="coerce").dropna()
print("linked manufacturer rows:", len(vendor_codes))
print("  as text, found in vendor file:", maint_raw["VendorCode"].isin(vendors_raw["VendorId"]).sum())
print("  as numbers, found in vendor file:", vendor_codes.isin(vendor_ids).sum())

linked manufacturer rows: 11982
  as text, found in vendor file: 0
  as numbers, found in vendor file: 11973


In [12]:
# the most linked vendors; 5 and 2 exist but have no name (so does 0)
top = vendor_codes.value_counts().head(6).rename_axis("vendor_code").reset_index(name="manufacturers")
names = vendors_raw.assign(vendor_code=vendor_ids).dropna(subset=["vendor_code"]).set_index("vendor_code")["VendorName"]
top["VendorName"] = top["vendor_code"].map(names)
top

,vendor_code,manufacturers,VendorName
0,5.0,3405,NaN
1,3197.0,250,MASLINE ELECTRONICS INC.
2,100078.0,124,PIONEER INDUSTRIES INC.
3,2.0,64,NaN
4,109.0,48,HAUN WELDING SUPPLY
5,5974.0,44,"WESCO DISTRIBUTION, INC."


### Cages table
Our reference table: one row per cage code with the official company name.

In [13]:
print("cage ids unique:", cages_raw["id"].is_unique)
print("cage id lengths:", cages_raw["id"].str.len().value_counts().to_dict())
print("missing company names:", cages_raw["company"].isna().sum())
cages_raw["status"].value_counts().to_frame()

cage ids unique: True
cage id lengths: {5: 4005840}


missing company names: 301


,count
status,
A,2837931
H,534696
N,321714
P,135274
R,113351
F,28645
C,18047
Y,11577
E,3544


In [14]:
maint_cages = maint_raw["CageCode"].str.strip().str.upper()
print("maintenance cage codes found in cages table:", maint_cages.isin(cages_raw["id"]).sum(), "of", maint_cages.notna().sum())

maintenance cage codes found in cages table: 20007 of 20384


### What this means for the cleaning

- Manufacturer names come from the cages table, looked up by cage code.
- Vendors need their cage found by name, so names have to be normalised before matching.
- Vendor ids and vendor codes are compared as numbers.
- Vendors 0, 2 and 5 have no name, so the manufacturers pointing at them have no real parent and are treated as orphans.
- Emails are pulled out of `Description`; fax numbers and notes there are ignored.